In [1]:
import pandas as pd
import geopandas as gpd
from libpysal.weights import DistanceBand
import numpy as np
import statsmodels.api as sm
from pysal.model.spreg import ML_Error
from pysal.model.spreg import ML_Lag
from libpysal.weights import W
import numpy as np
from libpysal.weights import WSP
import scipy.sparse as sp
from libpysal.weights import Queen
from shapely.geometry import Polygon
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('post_processed_data.csv')
gdf = gpd.read_file('/Polygon/county.shp')

In [4]:
gdf = gdf[['FIPS','geometry']].copy()
gdf.FIPS = gdf.FIPS.astype(int)
gdf = gdf.merge(df,on='FIPS',how='right')
gdf['unique_id'] = gdf['FIPS'].astype(str) + "_" + gdf['storm'] + "_" + gdf['order'].astype(str)
data = gdf.copy()

In [5]:
# create weight matrix 
w = Queen.from_dataframe(gdf)
adj_matrix = pd.DataFrame(w.full()[0], index=gdf['unique_id'], columns=gdf['unique_id'])

storm_matrix = pd.DataFrame(
    [[1 if gdf.loc[i, 'storm'] == gdf.loc[j, 'storm'] else 0 
      for j in range(len(gdf))]
     for i in range(len(gdf))],
    index=gdf['unique_id'], columns=gdf['unique_id']
)
filtered_adj_matrix = adj_matrix * storm_matrix

A = filtered_adj_matrix.to_numpy()   
row_sums = A.sum(axis=1, keepdims=True) 
W = np.divide(A, row_sums, out=np.zeros_like(A), where=row_sums!=0)
np.fill_diagonal(W, 0.0)  
w = WSP(sp.csr_matrix(W)).to_W()

data = data.reset_index()
data.index = range(len(data))
w.id_order = list(data.index)

/var/folders/ch/jnm4tqzx5vs9wyxp0ls6q_br0000gp/T/ipykernel_28366/2679773881.py:2: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)
/Applications/anaconda3/envs/Jan26/lib/python3.11/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)
/Applications/anaconda3/envs/Jan26/lib/python3.11/site-packages/libpysal/weights/weights.py:1685: UserWarning: The weights matrix is not fully connected: 
 There are 11 disconnected components.
  w = W(neighbors, weights, ids, silence_warnings=silence_warnings)


## Gasoline Stations

In [6]:
X = data[[
       'order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Gasoline Stations density', 
       'state_LA', 'state_NC', 'state_NJ', 'state_NY', 'state_SC', 'state_TX',
]]
continuous_vars = ['order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Gasoline Stations density'] 
dummy_vars = [col for col in data.columns if col.startswith('state_')]
X_continuous = data[continuous_vars]
X_dummy = data[dummy_vars]
X = pd.concat([X_continuous, X_dummy], axis=1)
y = data['pre_Gasoline Stations']

### OLS

In [7]:
from spreg import OLS
ols_model = OLS(y, X, w=w ,spat_diag=True,moran=True)
print(ols_model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Gasoline Stations                Number of Observations:        1360
Mean dependent var  :      1.0454                Number of Variables   :          18
S.D. dependent var  :      0.1128                Degrees of Freedom    :        1342
R-squared           :      0.2796
Adjusted R-squared  :      0.2705
Sum squared residual:     12.4531                F-statistic           :     30.6394
Sigma-square        :       0.009                Prob(F-statistic)     :   2.318e-83
S.E. of regression  :       0.096                Log likelihood        :    1261.670
Sigma-square ML     :       0.009                Akaike info criterion :   -2487.340
S.E of regression ML:      0.0957                Schwarz criterion     :   -2393.466

---------------------------------------------------

### SEM

In [8]:
model_sem = ML_Error(y, X, w)
print(model_sem.summary)

/Applications/anaconda3/envs/Jan26/lib/python3.11/site-packages/spreg/ml_error.py:183: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
---------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Gasoline Stations                Number of Observations:        1360
Mean dependent var  :      1.0454                Number of Variables   :          18
S.D. dependent var  :      0.1128                Degrees of Freedom    :        1342
Pseudo R-squared    :      0.2766
Log likelihood      :   1336.8644
Sigma-square ML     :      0.0078                Akaike info criterion :   -2637.729
S.E of regression   :      0.0886                Schwarz criterion     :   -2543.854

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.

### SAM 

In [9]:
model_slm = ML_Lag(y, X, w)
print(model_slm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Gasoline Stations                Number of Observations:        1360
Mean dependent var  :      1.0454                Number of Variables   :          19
S.D. dependent var  :      0.1128                Degrees of Freedom    :        1341
Pseudo R-squared    :      0.3691
Spatial Pseudo R-squared:  0.2681
Log likelihood      :   1326.2340
Sigma-square ML     :      0.0080                Akaike info criterion :   -2614.468
S.E of regression   :      0.0897                Schwarz criterion     :   -2515.378

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------

### SDM

In [10]:
X1 = X[['fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Gasoline Stations density']].copy()

In [11]:
X_lagged = w.sparse @ X1.values
X_lagged = pd.DataFrame(X_lagged, 
                        columns=[f"lagged_{col}" for col in X1.columns], 
                        index=X_continuous.index)

X = pd.concat([X_continuous, X_dummy, X_lagged], axis=1)
y = data['pre_Gasoline Stations']
model_sdm = ML_Lag(y, X, w)
print(model_sdm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Gasoline Stations                Number of Observations:        1360
Mean dependent var  :      1.0454                Number of Variables   :          27
S.D. dependent var  :      0.1128                Degrees of Freedom    :        1333
Pseudo R-squared    :      0.3894
Spatial Pseudo R-squared:  0.2860
Log likelihood      :   1342.4226
Sigma-square ML     :      0.0078                Akaike info criterion :   -2630.845
S.E of regression   :      0.0883                Schwarz criterion     :   -2490.034

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------

## Grocery Stores

In [12]:
X = data[[
       'order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Grocery Stores density', 
       'state_LA', 'state_NC', 'state_NJ', 'state_NY', 'state_SC', 'state_TX',
]]
y = data['pre_Grocery Stores']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:     pre_Grocery Stores   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.037
Method:                 Least Squares   F-statistic:                     4.109
Date:                Tue, 07 Oct 2025   Prob (F-statistic):           3.94e-08
Time:                        23:08:52   Log-Likelihood:                -549.50
No. Observations:                1360   AIC:                             1135.
Df Residuals:                    1342   BIC:                             1229.
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      1

## Spatial Model

In [13]:
from spreg import OLS
ols1 = OLS(y,X,w=w,spat_diag=True,moran=True)
print(ols1.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Grocery Stores                Number of Observations:        1360
Mean dependent var  :      1.0615                Number of Variables   :          18
S.D. dependent var  :      0.3719                Degrees of Freedom    :        1342
R-squared           :      0.0495
Adjusted R-squared  :      0.0374
Sum squared residual:     178.654                F-statistic           :      4.1090
Sigma-square        :       0.133                Prob(F-statistic)     :   3.943e-08
S.E. of regression  :       0.365                Log likelihood        :    -549.499
Sigma-square ML     :       0.131                Akaike info criterion :    1134.997
S.E of regression ML:      0.3624                Schwarz criterion     :    1228.871

------------------------------------------------------

In [14]:
continuous_vars = ['order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Grocery Stores density'] 
dummy_vars = [col for col in data.columns if col.startswith('state_')]
X_continuous = data[continuous_vars]
X_dummy = data[dummy_vars]
X = pd.concat([X_continuous, X_dummy], axis=1)
y = data['pre_Grocery Stores']

### OLS

In [15]:
ols_model = OLS(y, X, w=w)
print(ols_model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Grocery Stores                Number of Observations:        1360
Mean dependent var  :      1.0615                Number of Variables   :          18
S.D. dependent var  :      0.3719                Degrees of Freedom    :        1342
R-squared           :      0.0495
Adjusted R-squared  :      0.0374
Sum squared residual:     178.654                F-statistic           :      4.1090
Sigma-square        :       0.133                Prob(F-statistic)     :   3.943e-08
S.E. of regression  :       0.365                Log likelihood        :    -549.499
Sigma-square ML     :       0.131                Akaike info criterion :    1134.997
S.E of regression ML:      0.3624                Schwarz criterion     :    1228.871

------------------------------------------------------

### SEM

In [16]:
model_sem = ML_Error(y, X, w)
print(model_sem.summary)

/Applications/anaconda3/envs/Jan26/lib/python3.11/site-packages/spreg/ml_error.py:183: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
---------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Grocery Stores                Number of Observations:        1360
Mean dependent var  :      1.0615                Number of Variables   :          18
S.D. dependent var  :      0.3719                Degrees of Freedom    :        1342
Pseudo R-squared    :      0.0495
Log likelihood      :   -549.4767
Sigma-square ML     :      0.1314                Akaike info criterion :    1134.953
S.E of regression   :      0.3624                Schwarz criterion     :    1228.828

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.273

### SAM 

In [17]:
model_slm = ML_Lag(y, X, w)
print(model_slm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Grocery Stores                Number of Observations:        1360
Mean dependent var  :      1.0615                Number of Variables   :          19
S.D. dependent var  :      0.3719                Degrees of Freedom    :        1341
Pseudo R-squared    :      0.0495
Spatial Pseudo R-squared:  0.0495
Log likelihood      :   -549.4778
Sigma-square ML     :      0.1314                Akaike info criterion :    1136.956
S.E of regression   :      0.3624                Schwarz criterion     :    1236.045

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
---------------------------------------------------------

### SDM

In [18]:
X1 = X[[
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Grocery Stores density']].copy()

In [19]:
X_lagged = w.sparse @ X1.values
X_lagged = pd.DataFrame(X_lagged, 
                        columns=[f"lagged_{col}" for col in X1.columns], 
                        index=X_continuous.index)

X = pd.concat([X_continuous, X_dummy, X_lagged], axis=1)
y = data['pre_Grocery Stores']
model_sdm = ML_Lag(y, X, w)
print(model_sdm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Grocery Stores                Number of Observations:        1360
Mean dependent var  :      1.0615                Number of Variables   :          27
S.D. dependent var  :      0.3719                Degrees of Freedom    :        1333
Pseudo R-squared    :      0.0756
Spatial Pseudo R-squared:  0.0753
Log likelihood      :   -530.5610
Sigma-square ML     :      0.1277                Akaike info criterion :    1115.122
S.E of regression   :      0.3574                Schwarz criterion     :    1255.934

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
---------------------------------------------------------

## Building Dealers

In [20]:
X = data[[
       'order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Building Dealers density', 
       'state_LA', 'state_NC', 'state_NJ', 'state_NY', 'state_SC', 'state_TX',
]]
continuous_vars = ['order', 'forecast_wind', 'order_x_wind',
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Building Dealers density'] 
dummy_vars = [col for col in data.columns if col.startswith('state_')]
X_continuous = data[continuous_vars]
X_dummy = data[dummy_vars]
X = pd.concat([X_continuous, X_dummy], axis=1)
y = data['pre_Building Dealers']

## Spatial Model

In [21]:
from spreg import OLS
ols1 = OLS(y,X,w=w)
print(ols1.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Building Dealers                Number of Observations:        1360
Mean dependent var  :      1.0259                Number of Variables   :          18
S.D. dependent var  :      0.3185                Degrees of Freedom    :        1342
R-squared           :      0.0858
Adjusted R-squared  :      0.0743
Sum squared residual:     126.057                F-statistic           :      7.4118
Sigma-square        :       0.094                Prob(F-statistic)     :   9.524e-18
S.E. of regression  :       0.306                Log likelihood        :    -312.371
Sigma-square ML     :       0.093                Akaike info criterion :     660.741
S.E of regression ML:      0.3044                Schwarz criterion     :     754.616

----------------------------------------------------

### OLS

In [22]:
ols_model = OLS(y, X, w=w)
print(ols_model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Building Dealers                Number of Observations:        1360
Mean dependent var  :      1.0259                Number of Variables   :          18
S.D. dependent var  :      0.3185                Degrees of Freedom    :        1342
R-squared           :      0.0858
Adjusted R-squared  :      0.0743
Sum squared residual:     126.057                F-statistic           :      7.4118
Sigma-square        :       0.094                Prob(F-statistic)     :   9.524e-18
S.E. of regression  :       0.306                Log likelihood        :    -312.371
Sigma-square ML     :       0.093                Akaike info criterion :     660.741
S.E of regression ML:      0.3044                Schwarz criterion     :     754.616

----------------------------------------------------

### SEM

In [23]:
model_sem = ML_Error(y, X, w)
print(model_sem.summary)

/Applications/anaconda3/envs/Jan26/lib/python3.11/site-packages/spreg/ml_error.py:183: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
---------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Building Dealers                Number of Observations:        1360
Mean dependent var  :      1.0259                Number of Variables   :          18
S.D. dependent var  :      0.3185                Degrees of Freedom    :        1342
Pseudo R-squared    :      0.0857
Log likelihood      :   -306.9275
Sigma-square ML     :      0.0916                Akaike info criterion :     649.855
S.E of regression   :      0.3026                Schwarz criterion     :     743.729

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.4

### SAM 

In [24]:
model_slm = ML_Lag(y, X, w)
print(model_slm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Building Dealers                Number of Observations:        1360
Mean dependent var  :      1.0259                Number of Variables   :          19
S.D. dependent var  :      0.3185                Degrees of Freedom    :        1341
Pseudo R-squared    :      0.0977
Spatial Pseudo R-squared:  0.0869
Log likelihood      :   -306.3354
Sigma-square ML     :      0.0915                Akaike info criterion :     650.671
S.E of regression   :      0.3025                Schwarz criterion     :     749.760

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
-------------------------------------------------------

### SDM

In [25]:
X1 = X[[
       'fraction_pop_under_5', 'fraction_pop_over_65', 'dis_5to64_rate',
       'lmt_english', 'no_vehicle_rate', 'median_income', 'white_fraction',
       'Building Dealers density']].copy()

In [26]:
X_lagged = w.sparse @ X1.values
X_lagged = pd.DataFrame(X_lagged, 
                        columns=[f"lagged_{col}" for col in X1.columns], 
                        index=X_continuous.index)

X = pd.concat([X_continuous, X_dummy, X_lagged], axis=1)
y = data['pre_Building Dealers']
model_sdm = ML_Lag(y, X, w)
print(model_sdm.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
-----------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :pre_Building Dealers                Number of Observations:        1360
Mean dependent var  :      1.0259                Number of Variables   :          27
S.D. dependent var  :      0.3185                Degrees of Freedom    :        1333
Pseudo R-squared    :      0.1031
Spatial Pseudo R-squared:  0.0933
Log likelihood      :   -301.9033
Sigma-square ML     :      0.0910                Akaike info criterion :     657.807
S.E of regression   :      0.3016                Schwarz criterion     :     798.618

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
-------------------------------------------------------